# 03-7. 仮説検定 — 動かして確かめる

📖 解説: [`../07_hypothesis_testing.md`](../07_hypothesis_testing.md)

上から順に **Shift+Enter** で実行していってください。

## このノートで触るもの
1. まったく同じ 2 つの案でも「差」は出る — 帰無分布を見る
2. $t$ 検定を実行する
3. ⚠️ p 値の分布 — $H_0$ が正しいとき p 値は一様分布になる
4. 【対話】検出力 — 差があるのに見逃す確率
5. ⚠️ $n$ を増やせば、どんな微小な差も「有意」になる
6. ⚠️ p ハッキング — 有意になるまでデータを足すとどうなるか
7. 多重比較 — 20 回検定すれば 1 回は当たる

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (07_hypothesis_testing.md)](../07_hypothesis_testing.md)

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, IntSlider, FloatSlider

%matplotlib inline

rng = np.random.default_rng(42)

## 1. まったく同じ 2 案でも「差」は出る

A 案と B 案が**本当にまったく同じ**（$H_0$ が真）だとします。
それでも、測るたびに差はゼロにはなりません。

まず「偶然だけで、どれくらいの差が出るものなのか」を体感します。

In [ ]:
N: int = 30            # 各群のサンプルサイズ
TRUE_MEAN: float = 50.0
TRUE_SD: float = 5.0

# H0 が真 (2 群はまったく同じ分布) の状況を 10000 回シミュレーション
sim = np.random.default_rng(0)
n_sim: int = 10000
a = sim.normal(TRUE_MEAN, TRUE_SD, size=(n_sim, N))   # shape: (10000, 30)
b = sim.normal(TRUE_MEAN, TRUE_SD, size=(n_sim, N))   # 同じ分布!
diffs = a.mean(axis=1) - b.mean(axis=1)               # shape: (10000,)

plt.figure(figsize=(9, 4))
plt.hist(diffs, bins=60, density=True, alpha=0.75)
plt.axvline(0, c='k', lw=2, label='真の差 = 0')
for q, c in [(2.5, 'r'), (97.5, 'r')]:
    plt.axvline(np.percentile(diffs, q), c=c, ls='--', lw=1.5)
plt.xlabel('A の平均 − B の平均'); plt.ylabel('密度')
plt.title('2 群がまったく同じでも、これだけ差は出る (帰無分布)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'差はゼロが中心だが、実際には {diffs.min():.2f} 〜 {diffs.max():.2f} まで散らばる')
print(f'95% は {lo:.2f} 〜 {hi:.2f} の範囲に収まる (赤い破線)')
print()
print(f'例えば「差が {hi:.2f} 以上」だったら、偶然にしては珍しい (片側 2.5%)。')
print('この「珍しさ」を数値にしたものが p 値。')

## 2. t 検定を実行する

$$
t = \frac{\text{観測された差}}{\text{差の標準誤差}}
$$

「差を、ばらつきで割る」— これが検定統計量の基本形です。

In [ ]:
# 今度は本当に差がある 2 群 (真の差は 2.0 秒)
group_a: np.ndarray = rng.normal(loc=50.0, scale=5.0, size=30)   # shape: (30,)
group_b: np.ndarray = rng.normal(loc=52.0, scale=5.0, size=30)   # shape: (30,)

t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)  # Welch

print(f'A の平均 : {group_a.mean():.2f} 秒 (SD {group_a.std(ddof=1):.2f})')
print(f'B の平均 : {group_b.mean():.2f} 秒 (SD {group_b.std(ddof=1):.2f})')
print(f'観測された差: {group_b.mean() - group_a.mean():.2f} 秒')
print()
print(f't 統計量 : {t_stat:.4f}')
print(f'p 値     : {p_value:.4f}')
print(f'判定     : {"有意差あり" if p_value < 0.05 else "有意差なし"} (α=0.05)')
print()
# 効果量と信頼区間も必ず併記する
pooled_sd = np.sqrt((group_a.var(ddof=1) + group_b.var(ddof=1)) / 2)
cohens_d = (group_b.mean() - group_a.mean()) / pooled_sd
se_diff = np.sqrt(group_a.var(ddof=1)/len(group_a) + group_b.var(ddof=1)/len(group_b))
df = len(group_a) + len(group_b) - 2
ci = stats.t.ppf(0.975, df) * se_diff
print(f'効果量 (Cohen\'s d) : {cohens_d:.3f}')
print(f'差の 95% 信頼区間   : [{group_b.mean()-group_a.mean()-ci:.2f}, '
      f'{group_b.mean()-group_a.mean()+ci:.2f}] 秒')
print()
print('★ p 値だけでなく、効果量と信頼区間をセットで報告するのが作法。')

## 3. ⚠️ p 値の分布 — $H_0$ が正しいとき p 値は一様分布になる

これを知っておくと、p 値の意味が腑に落ちます。

**$H_0$ が正しい（差がない）とき、p 値は 0〜1 の一様分布**になります。
つまり「$p < 0.05$」は、差がなくても **5% の確率で起きる**。それが第一種の過誤です。

In [ ]:
# H0 が真のとき (差なし) と、H1 が真のとき (差あり) で p 値の分布を比べる
def p_value_distribution(n_sim: int = 5000, n: int = 30, true_diff: float = 0.0) -> np.ndarray:
    """2 標本 t 検定を n_sim 回繰り返して p 値の分布を得る.

    Args:
        n_sim: シミュレーション回数
        n: 各群のサンプルサイズ
        true_diff: 母平均の真の差

    Returns:
        p 値の配列, shape: (n_sim,)
    """
    s = np.random.default_rng(1)
    xa = s.normal(50.0, 5.0, size=(n_sim, n))
    xb = s.normal(50.0 + true_diff, 5.0, size=(n_sim, n))
    _, p = stats.ttest_ind(xa, xb, axis=1, equal_var=False)
    return p


p_null = p_value_distribution(true_diff=0.0)   # H0 が真
p_alt = p_value_distribution(true_diff=3.0)    # H1 が真 (差 3.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(p_null, bins=20, range=(0, 1), density=True, alpha=0.8, color='tab:blue')
axes[0].axhline(1.0, c='k', ls='--', label='一様分布')
axes[0].axvline(0.05, c='r', lw=2, label='α=0.05')
axes[0].set_title(f'H₀ が真 (差なし)\np<0.05 の割合 = {(p_null<0.05).mean():.3f}')
axes[0].set_xlabel('p 値'); axes[0].legend(fontsize=8)

axes[1].hist(p_alt, bins=20, range=(0, 1), density=True, alpha=0.8, color='tab:green')
axes[1].axvline(0.05, c='r', lw=2, label='α=0.05')
axes[1].set_title(f'H₁ が真 (差 3.0)\np<0.05 の割合 = {(p_alt<0.05).mean():.3f} = 検出力')
axes[1].set_xlabel('p 値'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f'H₀ が真のとき p<0.05 になる割合: {(p_null<0.05).mean():.4f}  ← ほぼ 0.05 (=α)')
print('  → 差がなくても 20 回に 1 回は「有意差あり」と出る。これが第一種の過誤。')
print()
print(f'H₁ が真のとき p<0.05 になる割合: {(p_alt<0.05).mean():.4f}  ← これが検出力 (1-β)')
print(f'  → 逆に言うと {1-(p_alt<0.05).mean():.1%} は差を見逃している (第二種の過誤)。')

## 4. 【対話】検出力 — 見逃しの確率

**注目点**: 真の差が小さいほど、$n$ が少ないほど、検出力は下がります。
慣習的な目安は **検出力 0.8 以上**。それを満たす $n$ を事前に決めるのが調査設計です。

In [ ]:
def power_analysis(n: int = 30, true_diff: float = 2.0) -> None:
    """サンプルサイズと真の差から検出力をシミュレーションで求める.

    Args:
        n: 各群のサンプルサイズ
        true_diff: 母平均の真の差
    """
    n_sim: int = 3000
    s = np.random.default_rng(2)
    xa = s.normal(50.0, 5.0, size=(n_sim, n))
    xb = s.normal(50.0 + true_diff, 5.0, size=(n_sim, n))
    _, p = stats.ttest_ind(xa, xb, axis=1, equal_var=False)
    power = (p < 0.05).mean()

    plt.figure(figsize=(9, 3.5))
    plt.hist(p, bins=40, range=(0, 1), alpha=0.8,
             color='tab:green' if power >= 0.8 else 'tab:orange')
    plt.axvline(0.05, c='r', lw=2, label='α=0.05')
    plt.xlabel('p 値'); plt.ylabel('回数')
    plt.title(f'n={n} / 真の差={true_diff}:  検出力 = {power:.3f}'
              f'{"  ✅ 十分" if power >= 0.8 else "  ⚠️ 不足"}')
    plt.legend(); plt.show()

    print(f'検出力 (1-β) = {power:.3f}   見逃す確率 (β) = {1-power:.3f}')
    if power < 0.8:
        print('→ 検出力不足。差があっても見つけられない可能性が高い。')
        print('  「有意差なし」が「差がない」を意味しないのは、こういう場合があるから。')


interact(power_analysis,
         n=IntSlider(min=5, max=300, step=5, value=30),
         true_diff=FloatSlider(min=0.0, max=5.0, step=0.25, value=2.0))

## 5. ⚠️ $n$ を増やせば、どんな微小な差も「有意」になる

$t = \dfrac{\text{差}}{\text{SE}}$ の分母には $\sqrt{n}$ が入っています。
つまり **$n$ を大きくすれば分母が小さくなり、$t$ はいくらでも大きくできます**。

クリック率 **10.0% 対 10.1%**（差 0.1 ポイント）を固定して、$n$ だけ変えてみます。

In [ ]:
def prop_z_test(p1: float, p2: float, n: int) -> tuple[float, float]:
    """2 標本比率の差の z 検定 (両側).

    Args:
        p1: 群 1 の比率
        p2: 群 2 の比率
        n: 各群のサンプルサイズ

    Returns:
        (z 統計量, 両側 p 値)
    """
    pool = (p1 + p2) / 2
    se = np.sqrt(pool * (1 - pool) * (2 / n))
    z = (p1 - p2) / se
    return z, 2 * (1 - stats.norm.cdf(abs(z)))


print('クリック率 10.0% vs 10.1% (差はずっと 0.1 ポイントのまま)')
print()
print(f'{"n (各群)":>12}{"z":>10}{"p 値":>10}   判定')
print('-' * 46)
ns, ps = [], []
for n in (1_000, 10_000, 100_000, 1_000_000, 10_000_000):
    z, pv = prop_z_test(0.100, 0.101, n)
    ns.append(n); ps.append(pv)
    print(f'{n:>12,}{z:>10.3f}{pv:>10.4f}   {"★ 有意差あり" if pv < 0.05 else "有意差なし"}')

plt.figure(figsize=(9, 3.5))
plt.semilogx(ns, ps, 'o-', lw=2)
plt.axhline(0.05, c='r', ls='--', lw=2, label='α=0.05')
plt.xlabel('各群のサンプルサイズ n (対数目盛)'); plt.ylabel('p 値')
plt.title('差は同じ 0.1 ポイントなのに、n を増やすだけで「有意」になる')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print()
print('★ 統計的有意性は「偶然ではなさそう」としか言っていない。')
print('  クリック率が 0.1 ポイント違うことに意味があるかは、統計ではなくビジネスが決める。')

## 6. ⚠️ p ハッキング — 有意になるまでデータを足す

「まだ有意じゃないから、もう少しデータを集めよう」

これは**やってはいけない**分析です。どれくらい危険か、実験してみます。

**設定**: 2 群は**まったく同じ**（真の差はゼロ）。
それでも「有意になるまでデータを追加し続ける」とどうなるか。

In [ ]:
def p_hacking_trial(max_n: int = 200, check_every: int = 5, seed: int = 0) -> tuple[bool, int, list]:
    """差がない 2 群に対し、有意になるまでデータを追加し続ける.

    Args:
        max_n: 各群の最大サンプルサイズ
        check_every: 何件追加するごとに検定するか
        seed: 乱数シード

    Returns:
        (有意になったか, なった時点の n, p 値の推移)
    """
    s = np.random.default_rng(seed)
    xa = s.normal(50.0, 5.0, max_n)     # 2 群はまったく同じ分布
    xb = s.normal(50.0, 5.0, max_n)     # 真の差 = 0

    p_track = []
    for n in range(10, max_n + 1, check_every):
        _, p = stats.ttest_ind(xa[:n], xb[:n], equal_var=False)
        p_track.append((n, p))
        if p < 0.05:
            return True, n, p_track
    return False, max_n, p_track


# 1 回分の様子を描く
found, stop_n, track = p_hacking_trial(seed=11)
ns_t = [t[0] for t in track]; ps_t = [t[1] for t in track]

plt.figure(figsize=(9, 3.8))
plt.plot(ns_t, ps_t, 'o-', ms=4)
plt.axhline(0.05, c='r', ls='--', lw=2, label='α=0.05')
if found:
    plt.axvline(stop_n, c='g', lw=2, label=f'n={stop_n} で「有意」→ ここで止める誘惑')
plt.xlabel('各群のサンプルサイズ n'); plt.ylabel('p 値')
plt.title('真の差がゼロなのに、p 値は上下する')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

# 1000 回繰り返して、何%が「有意」にたどり着くか
hits = sum(p_hacking_trial(seed=s)[0] for s in range(1000))
print(f'真の差がゼロの 2 群に対して、有意になるまでデータを追加した結果:')
print(f'  1000 回中 {hits} 回 ({hits/10:.1f}%) が「有意差あり」に到達')
print()
print(f'★ 本来 α=5% のはずが、{hits/10:.1f}% まで跳ね上がった。')
print('  「そのうち当たる」まで続ければ、必ず当たる。これが p ハッキング。')
print()
print('対策: サンプルサイズを事前に決める / 分析計画を事前登録する /')
print('      途中で覗くなら逐次検定の補正を入れる')

## 7. 多重比較 — 20 回検定すれば 1 回は当たる

同じ理屈で、**たくさんの指標を検定して「当たった」ものだけ報告する**のも危険です。

In [ ]:
n_metrics: int = 20        # 20 個の指標を同時に検定する
n_trials: int = 2000
s = np.random.default_rng(5)

any_significant = 0
for _ in range(n_trials):
    # すべての指標で真の差はゼロ
    xa = s.normal(0, 1, size=(n_metrics, 30))
    xb = s.normal(0, 1, size=(n_metrics, 30))
    _, pvals = stats.ttest_ind(xa, xb, axis=1, equal_var=False)
    if (pvals < 0.05).any():
        any_significant += 1

rate = any_significant / n_trials
theory = 1 - 0.95 ** n_metrics

print(f'20 個の指標すべてで真の差がゼロなのに…')
print(f'  「少なくとも 1 つが有意」になった割合: {rate:.3f}')
print(f'  理論値 1 - 0.95^20              : {theory:.3f}')
print()
print(f'★ 20 個検定すれば {theory:.0%} の確率で「何か」が有意になる。')
print('  当たったものだけ報告すれば、いくらでも「発見」を作れてしまう。')
print()
print('対策の例 — Bonferroni 補正: α を検定回数で割る')
print(f'  α = 0.05 / {n_metrics} = {0.05/n_metrics:.4f} を基準にする (保守的だが単純)')

## まとめ

- **同じ 2 群でも差は出る**。だから「偶然にしては珍しいか」を問うのが検定
- **p 値** = $H_0$ のもとで今回以上に極端な結果が出る確率。$P(H_0 \mid D)$ ではない
- **$H_0$ が真なら p 値は一様分布**。だから差がなくても 5% は「有意」になる
- **検出力**が低いと、差があっても見逃す。「有意差なし」= 「差がない」ではない
- **$n$ を増やせば微小な差も有意になる**。効果量と信頼区間を必ず併記する
- **p ハッキング**（有意になるまで追加）で第一種の過誤は 5% → 20% 以上に跳ね上がる
- **多重比較**: 20 個検定すれば 64% の確率で何かが当たる

この章の核心:

> **p 値は「偶然にしては珍しい」としか言っていない。**
> **「意味のある差か」を決めるのは、統計ではなくあなたのドメイン知識である。**

→ 次は [`08_bayesian_inference.ipynb`](08_bayesian_inference.ipynb) — ベイズ推論: p 値の代わりに「$\theta$ そのものの確率」を出す立場

頻度論では言えなかった「この区間に真値が入る確率は 95%」を、ベイズはそのまま言えます。

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`06_estimation.ipynb`](06_estimation.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`08_bayesian_inference.ipynb`](08_bayesian_inference.ipynb) |